# **1. Data Load**

In [1]:
import re
import pandas as pd
import numpy as np
import urllib.request
from collections import Counter
from sklearn.model_selection import train_test_split

In [2]:
# 데이터 불러오기
urllib.request.urlretrieve("https://raw.githubusercontent.com/ukairia777/tensorflow-nlp-tutorial/main/10.%20RNN%20Text%20Classification/dataset/naver_shopping.txt", filename="naver_shopping.txt")
total_data=pd.read_table("naver_shopping.txt", names=['ratings','reviews'])
display(total_data.head())
display(total_data['ratings'].value_counts())

,ratings,reviews
0,5,배공빠르고 굿
1,2,택배가 엉망이네용 저희집 밑에층에 말도없이 놔두고가고
2,5,아주좋아요 바지 정말 좋아서2개 더 구매했어요 이가격에 대박입니다. 바느질이 조금 ...
3,2,선물용으로 빨리 받아서 전달했어야 하는 상품이었는데 머그컵만 와서 당황했습니다. 전...
4,5,민트색상 예뻐요. 옆 손잡이는 거는 용도로도 사용되네요 ㅎㅎ


ratings
5    81177
2    63989
1    36048
4    18786
Name: count, dtype: int64

# **2.Train/Test Split**

In [3]:
# 감성분석 실행 / Ratings Label들 긍정, 부정으로 나누기
# 원본 ratings는 보존하고 긍정/부정 라벨 생성
total_data["label"] = (total_data["ratings"] > 3).astype(int)
display(total_data["label"].value_counts())
display(total_data.head())

# 중복 샘플들 제거
total_data.drop_duplicates(subset=['reviews'], inplace=True)
print('총 샘플의 수 :',len(total_data))

# 결측치 없는지 확인
print(total_data.isnull().values.any()) # 하나라도 true가 있을 때 확인

# Label 분포 확인
display(total_data.groupby('label').size().reset_index(name='count'))

label
0    100037
1     99963
Name: count, dtype: int64

,ratings,reviews,label
0,5,배공빠르고 굿,1
1,2,택배가 엉망이네용 저희집 밑에층에 말도없이 놔두고가고,0
2,5,아주좋아요 바지 정말 좋아서2개 더 구매했어요 이가격에 대박입니다. 바느질이 조금 ...,1
3,2,선물용으로 빨리 받아서 전달했어야 하는 상품이었는데 머그컵만 와서 당황했습니다. 전...,0
4,5,민트색상 예뻐요. 옆 손잡이는 거는 용도로도 사용되네요 ㅎㅎ,1


총 샘플의 수 : 199908
False


,label,count
0,0,99955
1,1,99953


# **3. Modeling**

In [ ]:
%pip install transformers==4.55.4 tokenizers==0.21.4 sentencepiece
from transformers import pipeline

print('transformers:', __import__('transformers').__version__)

# 분류 모델 정의
classifier = pipeline("text-classification", model="matthewburke/korean_sentiment")

# 판단 함수 작성
def pred_sentiment(text):

    preds = classifier(text, return_all_scores=True, top_k=None)
    best_preds = max(preds, key=lambda x: x['score'])
    if best_preds['label'] == 'LABEL_1':
        return 1
    else:
        return 0


# 실제 모델링(pipeline은 추론으로만 활용 -> 바로 test)
train_data, test_data = train_test_split(total_data, test_size=0.2, random_state=42)
test_data_sample = test_data[:1000]
test_data_sample['pred'] = test_data_sample['reviews'].apply(pred_sentiment)
display(test_data_sample.head())

# 정확도 계산 함수
def compute_accuracy(df):
    correct = (df['pred'] == df['label']).sum()
    total = len(df)
    return correct / total

acc = compute_accuracy(test_data_sample)
print('정확도(%):', acc * 100)

  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [54 lines of output]
      Rust not found, installing into a temporary directory
      Python reports SOABI: cp314-win_amd64
      Computed rustc target triple: x86_64-pc-windows-msvc
      Installation directory: C:\Users\\xec삤\xec냼誘\xbc\AppData\Local\puccinialin\puccinialin\Cache
      Rustup already downloaded
      Installing rust to C:\Users\\xec삤\xec냼誘\xbc\AppData\Local\puccinialin\puccinialin\Cache\rustup
      warn: it looks like you have an existing rustup settings file at:
      warn: C:\Users\\xec삤\xec냼誘\xbc\AppData\Local\puccinialin\puccinialin\Cache\rustup\settings.toml
      warn: installing msvc toolchain without its prerequisites
      info: profile set to minimal
      info: setting default host tuple to x86_64-pc-windows-msvc
      warn: Updating existing toolchain, profile choice will be ignored
      info: syncing channel updat

  Using cached transformers-4.46.3-py3-none-any.whl.metadata (44 kB)
  Using cached tokenizers-0.20.3.tar.gz (340 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached sentencepiece-0.2.2-cp314-cp314-win_amd64.whl.metadata (34 kB)
  Using cached filelock-3.32.6-py3-none-any.whl.metadata (2.0 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached packaging-26.3-py3-none-any.whl.metadata (3.5 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-win_amd64.whl.metadata (2.4 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached safetensors-

Device set to use cpu


,ratings,reviews,label,pred
193242,1,너무 낮고 솜도 적고 실망스럽습니다,0,0
125080,1,피부에 뾰루지가 많이 올라와요,0,0
122750,5,배송도 빠르네요 가격대비 좋은것 같아요~~~ 첨에는 힘들어하나 조금 지나니 잘 하네요,1,1
72927,5,재구매입니다. 핏도 좋고 착용감도 좋습니다.,1,1
83890,1,파손제품 온거 출장같다 오늘 받았는데 현재상황 장난하시는지 택배회사 항의하세요,0,0


정확도(%): 87.6


# **4.Advance**

In [8]:
%pip install transformers==4.55.4 tokenizers==0.21.4 sentencepiece

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import transformers
print('transformers:', transformers.__version__)

tokenizer = AutoTokenizer.from_pretrained(
    "eenzeenee/t5-base-korean-summarization",
    use_fast=True
)
model = AutoModelForSeq2SeqLM.from_pretrained(
    "eenzeenee/t5-base-korean-summarization"
)

def summarize(text):
    prefix = "summarize:"
    inputs = tokenizer(
        [prefix + text],
        max_length=512,
        truncation=True,
        return_tensors="pt"
    )

# 생성형 모델에만 쓰이는 generate() 메서드 사용
    output = model.generate(
        **inputs,
        num_beams=3,
        min_length=10,
        max_length=64
    )

    decoded_output = tokenizer.batch_decode(
        output,
        skip_special_tokens=True
    )[0]

    return decoded_output.strip()


text = '''배우 배수지가 매니지먼트 숲과 전속계약을 체결했다. 수지는 8일 자신의 인스타그램에 '데뷔 때 부터 함께해온 소속사 JYP와 계약기간을 마치고 오늘부터 새로운 소속사 매니지먼트 숲과 함께 하게 되었다'고 밝혔다. 이어 수지는 '연습생으로 시작해서, 데뷔하고 9년의 시간이 흐른 지금까지, JYP와 함께했던 여러 영광의 순간들이 스쳐지나간다'면서 '9년 동안 항상 옆에서 서포트 해주셨던 JYP 모든 직원분들께 진심으로 감사드린다'고 인사를 잊지 않았다. 2010년 걸그룹 '미쓰에이'로 데뷔한 배수지는 2011년 KBS2 드라마 '드림하이'로 첫 연기 활동을 시작했다. 2012년 영화 '건축학개론'을 통해 스크린 데뷔를 한 뒤 가수 활동과 연기 활동을 꾸준히 병행해 오고 있다. 매니지먼트 숲 관계자는 '배우 배수지의 장점과 매력을 극대화할 수 있는 작품 선택부터 국내외 활동, 가수로서의 솔로 활동까지 활발하게 이루어질 수 있도록 지원할 예정이다'고 전했다. 특히 올해는 작품을 통해 연기자 배수지로 대중들과 만날 예정이다. 현재 촬영 중인 SBS 드라마 '배가본드'는 민항 여객기 추락 사고에 연루된 한 남자가 은폐된 진실 속에서 찾아낸 거대한 국가 비리를 파헤치게 되는 과정을 담은 이야기다. 배수지는 국정원 블랙요원 고해리 역으로 출연하며, 뒤이어 영화 '백두산'에도 합류한다. 매니지먼트 숲은 공유, 공효진, 김재욱, 서현진, 이천희, 전도연, 정유미, 남지현, 최우식, 유민규, 이재준, 정가람, 전소니 등 소속되어 있다.'''
result = summarize(text)
print(result)


Note: you may need to restart the kernel to use updated packages.
transformers: 4.55.4
배우 배수지는 소속사 JYP와 계약기간을 마치고 새로운 소속사 매니지먼트 숲과 전속계약을 체결했다. 매니지먼트 숲 관계자는 배우 배수지의 장점과 매력을 극대화할 수 있는
